# E028 — Image flagship stress test at val time

Train E007 exactly as before (PCA 50 + LogReg + +All aug). At val time,
apply each of 7 stresses ONLY to val images and measure EER. Stresses
simulate the 'schválně zprasené' eval data Burget warned about: heavy
JPEG, heavy blur, rotation, occlusion, downsample, and all combined.

If 0.97% is genuine robustness, stress should only mildly degrade EER.
If it was overfitting, stress collapses performance.

In [ ]:
from pathlib import Path
import sys, io
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from scipy.ndimage import gaussian_filter, rotate as ndimage_rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DATA = Path('../data').resolve()
SEED = 67
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target')

In [ ]:
def find_png(stem, data_dir):
    for sf in ['target_train', 'target_dev', 'non_target_train', 'non_target_dev']:
        p = data_dir / sf / (stem + '.png')
        if p.exists():
            return p
    raise FileNotFoundError(stem)

def load_image(png_path):
    img = np.array(PILImage.open(png_path).convert('RGB'), dtype=np.float32)
    return img.mean(axis=2).flatten()

def load_images(df, data_dir):
    return np.stack([load_image(find_png(r['stem'], data_dir)) for _, r in df.iterrows()])

# E007 aug
def aug_flip(x):
    return x.reshape(80, 80)[:, ::-1].flatten()
def aug_brightness(x, rng):
    return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)
def aug_noise(x, rng, sigma=15.0):
    return np.clip(x + rng.normal(0, sigma, x.shape), 0, 255)

def augment_all(X, y, seed):
    rng = np.random.default_rng(seed)
    aug_X, aug_y = [], []
    for xi, yi in zip(X, y):
        aug_X.append(aug_flip(xi));             aug_y.append(yi)
        aug_X.append(aug_brightness(xi, rng));  aug_y.append(yi)
        aug_X.append(aug_noise(xi, rng));       aug_y.append(yi)
    return np.vstack([X, np.stack(aug_X)]), np.concatenate([y, np.array(aug_y)])

print('IO + E007 aug defined.')

In [ ]:
def stress_jpeg(x, quality=15):
    img = PILImage.fromarray(x.reshape(80, 80).astype(np.uint8), mode='L')
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(PILImage.open(buf).convert('L'), dtype=np.float32).flatten()

def stress_blur(x, sigma=3.0):
    return gaussian_filter(x.reshape(80, 80), sigma=sigma).flatten()

def stress_rotate(x, angle, rng):
    a = rng.uniform(-angle, angle)
    r = ndimage_rotate(x.reshape(80, 80), a, reshape=False, mode='reflect')
    return np.clip(r, 0, 255).flatten()

def stress_occlude(x, patch=20, rng=None):
    img = x.reshape(80, 80).copy()
    y = rng.integers(0, 80 - patch)
    xi = rng.integers(0, 80 - patch)
    img[y:y+patch, xi:xi+patch] = 0
    return img.flatten()

def stress_downsample(x):
    img = PILImage.fromarray(x.reshape(80, 80).astype(np.uint8), mode='L')
    down = img.resize((40, 40), PILImage.BILINEAR)
    up   = down.resize((80, 80), PILImage.BILINEAR)
    return np.array(up, dtype=np.float32).flatten()

def stress_all(x, rng):
    x = stress_jpeg(x, quality=15)
    x = stress_blur(x, sigma=3.0)
    x = stress_rotate(x, 15, rng)
    x = stress_occlude(x, 20, rng=rng)
    x = stress_downsample(x)
    return x

def apply_stress(X, kind, seed):
    rng = np.random.default_rng(seed)
    out = np.empty_like(X)
    for i, xi in enumerate(X):
        if kind == 'jpeg_q15':
            out[i] = stress_jpeg(xi, quality=15)
        elif kind == 'blur_s3':
            out[i] = stress_blur(xi, sigma=3.0)
        elif kind == 'rotate_15':
            out[i] = stress_rotate(xi, 15, rng)
        elif kind == 'rotate_25':
            out[i] = stress_rotate(xi, 25, rng)
        elif kind == 'occlude_20':
            out[i] = stress_occlude(xi, 20, rng=rng)
        elif kind == 'downsample':
            out[i] = stress_downsample(xi)
        elif kind == 'all':
            out[i] = stress_all(xi, rng)
        else:
            raise ValueError(kind)
    return out

STRESSES = {
    'clean':      'Clean (E007)',
    'jpeg_q15':   'Heavy JPEG q=15',
    'blur_s3':    'Heavy blur σ=3.0',
    'rotate_15':  'Rotation ±15°',
    'rotate_25':  'Rotation ±25° (severe)',
    'occlude_20': 'Occlusion 20×20',
    'downsample': 'Downsample 40→80',
    'all':        'All combined',
}
print('Stresses defined:', list(STRESSES.keys()))

## Reality check — visualize stressed samples

In [ ]:
viz_rows = [manifest[manifest.label==1].iloc[0], manifest[manifest.label==0].iloc[3]]
stress_keys = [k for k in STRESSES if k != 'clean']
fig, axes = plt.subplots(len(viz_rows), len(stress_keys)+1, figsize=(2*(len(stress_keys)+1), 2*len(viz_rows)+0.5))
if len(viz_rows) == 1:
    axes = axes[None, :]
for ri, row in enumerate(viz_rows):
    orig = load_image(find_png(row['stem'], DATA))
    axes[ri, 0].imshow(orig.reshape(80,80), cmap='gray', vmin=0, vmax=255)
    axes[ri, 0].set_title('Original' if ri == 0 else '', fontsize=9)
    axes[ri, 0].set_ylabel(f"{row['stem']}\n({'target' if row['label']==1 else 'non-target'})", fontsize=8)
    axes[ri, 0].set_xticks([]); axes[ri, 0].set_yticks([])
    for ci, k in enumerate(stress_keys, start=1):
        xs = apply_stress(orig[None, :], k, seed=42)[0]
        axes[ri, ci].imshow(xs.reshape(80,80), cmap='gray', vmin=0, vmax=255)
        if ri == 0:
            axes[ri, ci].set_title(STRESSES[k], fontsize=8)
        axes[ri, ci].axis('off')
plt.suptitle('E028 — stressed val samples (reality check)', fontsize=12)
plt.tight_layout()
plt.show()

## Cross-validation: E007 train → stressed val

In [ ]:
N_PCA = 50
C_LOGREG = 1.0

# Per fold: train E007 once, reuse on each stressed val
fold_models = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    y_train  = train_df['label'].to_numpy()
    y_val    = val_df['label'].to_numpy()

    X_train_orig = load_images(train_df, DATA)
    X_val_orig   = load_images(val_df,   DATA)
    X_train, y_train_aug = augment_all(X_train_orig, y_train, seed=SEED+fold_id)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train_s)
    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    clf.fit(X_train_pca, y_train_aug)

    fold_models.append({
        'fold_id': fold_id,
        'val_idx': val_idx,
        'X_val_orig': X_val_orig,
        'y_val': y_val,
        'scaler': scaler,
        'pca': pca,
        'clf': clf,
    })
    print(f'Fold {fold_id}: trained on {len(X_train)} aug samples, val={len(X_val_orig)}')

# For each stress + each fold, score
rows = []
for k, label in STRESSES.items():
    fold_eers = []
    fold_dcfs = []
    for fm in fold_models:
        if k == 'clean':
            X_val_stress = fm['X_val_orig']
        else:
            # seed by stress+fold so stresses are deterministic but differ per fold
            X_val_stress = apply_stress(fm['X_val_orig'], k, seed=SEED + 1000*fm['fold_id'] + hash(k) % 1000)
        Xs  = fm['scaler'].transform(X_val_stress)
        Xp  = fm['pca'].transform(Xs)
        s   = fm['clf'].decision_function(Xp)
        y_v = fm['y_val']
        eer, _ = compute_eer(s[y_v==1], s[y_v==0])
        dcf, _ = compute_min_dcf(s[y_v==1], s[y_v==0])
        fold_eers.append(eer*100)
        fold_dcfs.append(dcf)
    rows.append({
        'key': k,
        'label': label,
        'f0': fold_eers[0], 'f1': fold_eers[1], 'f2': fold_eers[2],
        'mean': np.mean(fold_eers), 'std': np.std(fold_eers),
        'dcf_mean': np.mean(fold_dcfs),
    })

results = pd.DataFrame(rows)
clean_mean = results[results.key=='clean']['mean'].iloc[0]
results['delta'] = results['mean'] - clean_mean
print()
print(f"{'Stress':<25} {'F0':>7} {'F1':>7} {'F2':>7} {'Mean':>8} {'Std':>7} {'Δ':>8} {'min-DCF':>9}")
print('-' * 80)
for _, r in results.iterrows():
    print(f"{r['label']:<25} {r['f0']:>7.2f} {r['f1']:>7.2f} {r['f2']:>7.2f} "
          f"{r['mean']:>8.2f} {r['std']:>7.2f} {r['delta']:>+8.2f} {r['dcf_mean']:>9.4f}")
results

## Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#95A5A6' if r['key']=='clean' else '#E74C3C' for _, r in results.iterrows()]
bars = ax.bar(range(len(results)), results['mean'], yerr=results['std'],
              color=colors, alpha=0.85, capsize=5)
for bar, (_, r) in zip(bars, results.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, r['mean']+r['std']+0.5,
            f"{r['mean']:.1f}±{r['std']:.1f}", ha='center', fontsize=8)
ax.axhline(clean_mean, color='#27AE60', ls='--', lw=2, label=f'E007 clean = {clean_mean:.2f}%')
ax.set_xticks(range(len(results)))
ax.set_xticklabels(results['label'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('EER [%] (val-time stress)')
ax.set_title('E028 — EER under val-time stress (train = E007 +All aug)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mat = results[['f0','f1','f2']].to_numpy()
fig, ax = plt.subplots(figsize=(6, 0.55*len(results)+1))
im = ax.imshow(mat, cmap='Reds', aspect='auto', vmin=0, vmax=max(mat.max(), 1))
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f'{mat[i,j]:.1f}', ha='center', va='center',
                color='white' if mat[i,j] > mat.max()/2 else 'black', fontsize=9)
ax.set_xticks([0,1,2]); ax.set_xticklabels(['Fold 0','Fold 1','Fold 2'])
ax.set_yticks(range(len(results))); ax.set_yticklabels(results['label'], fontsize=9)
plt.colorbar(im, ax=ax, label='EER [%]')
ax.set_title('Per-fold EER under stress')
plt.tight_layout()
plt.show()